In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# Reusable one-dimensional data generator
true_beta = np.array([1.5, 2.0])

def design(x):
    return np.column_stack([np.ones(len(x)), x])

def predict(beta, x):
    return design(np.asarray(x)) @ beta

def make_data(n_train=120, n_test=1000, ratio=0.15, distance=10.0, layout='one_sided', seed=0):
    local_rng = np.random.default_rng(seed)
    x_train = local_rng.uniform(-3, 3, n_train)
    y_train = predict(true_beta, x_train) + local_rng.normal(0, 0.45, n_train)
    x_test = local_rng.uniform(-3, 3, n_test)
    y_test = predict(true_beta, x_test) + local_rng.normal(0, 0.45, n_test)
    n_bad = int(n_train * ratio)
    bad = local_rng.choice(n_train, n_bad, replace=False)
    if layout == 'one_sided':
        y_train[bad] += distance
    elif layout == 'symmetric':
        y_train[bad] += local_rng.choice([-1, 1], n_bad) * distance
    elif layout == 'asymmetric':
        # Couple the shift to x: points on the left move up and points on the right move down.
        y_train[bad] += np.where(x_train[bad] < 0, distance, -distance)
    return x_train, y_train, x_test, y_test, bad

def make_two_lines(n_each=90, seed=0):
    local_rng = np.random.default_rng(seed)
    x1 = local_rng.uniform(-3, 3, n_each)
    x2 = local_rng.uniform(-3, 3, n_each)
    beta1 = np.array([2.0, 1.0])   # y = x + 2
    beta2 = np.array([8.0, -2.0])    # y = -2x + 8
    y1 = predict(beta1, x1) + local_rng.normal(0, 0.35, n_each)
    y2 = predict(beta2, x2) + local_rng.normal(0, 0.35, n_each)
    return x1, y1, x2, y2, beta1, beta2

def metrics(beta, x_test, y_test):
    return np.mean((predict(beta, x_test) - y_test) ** 2), np.linalg.norm(beta - true_beta)


In [ ]:
# Ordinary least squares baseline: solve the two normal equations by sums.
def ols_fit(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    sum_x = np.sum(x)
    sum_y = np.sum(y)
    sum_xx = np.sum(x * x)
    sum_xy = np.sum(x * y)
    determinant = n * sum_xx - sum_x ** 2
    if np.isclose(determinant, 0.0):
        raise ValueError('Cannot fit a line when x has no variation.')
    intercept = (sum_xx * sum_y - sum_x * sum_xy) / determinant
    slope = (n * sum_xy - sum_x * sum_y) / determinant
    return np.array([intercept, slope])


In [ ]:
# 1.345 is the conventional Huber tuning constant for about 95% Gaussian efficiency
# Huber downweights only residuals beyond the Huber threshold
def huber_weights(residual, c=1.345):
    scale = max(1.4826 * np.median(np.abs(residual - np.median(residual))), 1e-8)
    return np.minimum(1.0, c * scale / np.maximum(np.abs(residual), 1e-8))


# 2.3849 is the conventional Cauchy/Lorentzian tuning constant for about 95% Gaussian efficiency
def reweighting_weights(residual, c=2.3849):
    scale = max(1.4826 * np.median(np.abs(residual - np.median(residual))), 1e-8)
    return 1.0 / (1.0 + (residual / (c * scale)) ** 2)

def weighted_fit(x, y, weights):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    weights = np.asarray(weights, dtype=float)
    sum_w = np.sum(weights)
    sum_wx = np.sum(weights * x)
    sum_wy = np.sum(weights * y)
    sum_wxx = np.sum(weights * x * x)
    sum_wxy = np.sum(weights * x * y)
    determinant = sum_w * sum_wxx - sum_wx ** 2
    if np.isclose(determinant, 0.0):
        raise ValueError('Cannot fit a weighted line when x has no weighted variation.')
    intercept = (sum_wxx * sum_wy - sum_wx * sum_wxy) / determinant
    slope = (sum_w * sum_wxy - sum_wx * sum_wy) / determinant
    return np.array([intercept, slope])

def huber_fit(x, y, max_iter=100, tol=1e-8):
    beta = ols_fit(x, y)
    for _ in range(max_iter):
        updated = weighted_fit(x, y, huber_weights(y - predict(beta, x)))
        if np.linalg.norm(updated - beta) < tol:
            return updated
        beta = updated
    return beta


In [ ]:
# Repeatedly apply the same smooth residual weighting rule
def iterative_reweighted_fit(x, y, max_iter=100, tol=1e-8):
    beta = ols_fit(x, y)
    for _ in range(max_iter):
        updated = weighted_fit(x, y, reweighting_weights(y - predict(beta, x)))
        if np.linalg.norm(updated - beta) < tol:
            return updated
        beta = updated
    return beta


In [ ]:
# Fit once, then linearly downweight by absolute residual and refit once
def one_step_reweighted_fit(x, y):
    initial = ols_fit(x, y)
    distance = np.abs(y - predict(initial, x))
    farthest = np.max(distance)
    weights = 1.0 - distance / farthest if farthest > 0 else np.ones_like(distance)
    return weighted_fit(x, y, weights)

methods = {
    'OLS': ols_fit,
    'Huber': huber_fit,
    'Iterative reweighting': iterative_reweighted_fit,
    'One-step reweighting': one_step_reweighted_fit
}

# Test D reports OLS numerically but omits it from the distance-scan plot.
robust_methods = {name: fit for name, fit in methods.items() if name != 'OLS'}


In [ ]:
# Test A: vary only the proportion of contaminated observations
ratios = np.linspace(0, 0.4, 9)
records = []
for ratio in ratios:
    for seed in range(30):
        x_train, y_train, x_test, y_test, _ = make_data(ratio=ratio, distance=10, layout='one_sided', seed=seed)
        mse, parameter_error = metrics(ols_fit(x_train, y_train), x_test, y_test)
        records.append((ratio, mse, parameter_error))
a = pd.DataFrame(records, columns=['ratio', 'test_mse', 'parameter_error']).groupby('ratio').mean()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(a.index, a.test_mse, marker='o')
axes[1].plot(a.index, a.parameter_error, marker='o', color='tab:red')
axes[0].set(xlabel='Contamination ratio', ylabel='OLS test MSE', title='Test A')
axes[1].set(xlabel='Contamination ratio', ylabel='OLS parameter error', title='Test A')
plt.tight_layout()
plt.show()
print('Test A means over 30 random seeds:')
a


In [ ]:
# Test B: vary only how far contaminated observations are from the line
distances = np.linspace(0, 14, 10)
records = []
for distance in distances:
    for seed in range(30):
        x_train, y_train, x_test, y_test, _ = make_data(ratio=0.18, distance=distance, layout='one_sided', seed=seed)
        mse, parameter_error = metrics(ols_fit(x_train, y_train), x_test, y_test)
        records.append((distance, mse, parameter_error))
b = pd.DataFrame(records, columns=['distance', 'test_mse', 'parameter_error']).groupby('distance').mean()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(b.index, b.test_mse, marker='o')
axes[1].plot(b.index, b.parameter_error, marker='o', color='tab:red')
axes[0].set(xlabel='Outlier distance', ylabel='OLS test MSE', title='Test B')
axes[1].set(xlabel='Outlier distance', ylabel='OLS parameter error', title='Test B')
plt.tight_layout()
plt.show()
print('Test B means over 30 random seeds:')
b


In [ ]:
# Test C: compare numerical errors across contamination layouts
layouts = ['one_sided', 'symmetric', 'asymmetric']
summary = []
for layout in layouts:
    for seed in range(30):
        x_train, y_train, x_test, y_test, bad = make_data(ratio=0.18, distance=10, layout=layout, seed=seed)
        mse, parameter_error = metrics(ols_fit(x_train, y_train), x_test, y_test)
        summary.append((layout, mse, parameter_error))
c = pd.DataFrame(summary, columns=['layout', 'test_mse', 'parameter_error']).groupby('layout').mean().loc[layouts]
fig, ax = plt.subplots(figsize=(8, 4))
c.plot.bar(y=['test_mse', 'parameter_error'], ax=ax, rot=0)
ax.set(title='Test C errors', ylabel='Error')
plt.tight_layout()
plt.show()

# Show clean and contaminated points, the true line, and the OLS fit for each layout.
grid = np.linspace(-3.2, 3.2, 200)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, layout in zip(axes, layouts):
    x_train, y_train, _, _, bad = make_data(ratio=0.18, distance=10, layout=layout, seed=101)
    beta = ols_fit(x_train, y_train)
    clean = np.ones(len(x_train), dtype=bool)
    clean[bad] = False
    ax.scatter(x_train[clean], y_train[clean], s=18, alpha=0.7, label='Clean')
    ax.scatter(x_train[bad], y_train[bad], s=22, color='tab:red', label='Contamination')
    ax.plot(grid, predict(true_beta, grid), color='tab:blue', linewidth=2, label='Original line')
    ax.plot(grid, predict(beta, grid), color='black', linewidth=2, label='OLS fit')
    ax.set(title=layout, xlabel='x', ylabel='y')
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()
print('Test C means over 30 random seeds; scatter uses seed 101:')
c


In [ ]:
# Test D: first compare all four methods under clear, sparse contamination.
# Baseline: 18% of training responses shift upward by 14, versus noise SD 0.45.
baseline_ratio, baseline_distance, baseline_seed = 0.18, 14.0, 0
x_train, y_train, x_test, y_test, bad = make_data(ratio=baseline_ratio, distance=baseline_distance, layout='one_sided', seed=baseline_seed)
baseline_fits = {name: fit(x_train, y_train) for name, fit in methods.items()}
baseline_scores = pd.DataFrame(
    [(name, *metrics(beta, x_test, y_test)) for name, beta in baseline_fits.items()],
    columns=['method', 'test_mse', 'parameter_error']
).set_index('method')
grid = np.linspace(-3.2, 3.2, 200)
clean = np.ones(len(x_train), dtype=bool)
clean[bad] = False
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(x_train[clean], y_train[clean], s=18, alpha=0.6, label='Clean')
ax.scatter(x_train[bad], y_train[bad], s=24, color='tab:red', label='Contamination')
ax.plot(grid, predict(true_beta, grid), color='tab:blue', linewidth=2, label='Original line')
baseline_styles = {'OLS': ('black', '-'), 'Huber': ('tab:orange', '--'), 'Iterative reweighting': ('tab:green', '-.'), 'One-step reweighting': ('tab:purple', ':')}
for name, beta in baseline_fits.items():
    color, linestyle = baseline_styles[name]
    ax.plot(grid, predict(beta, grid), color=color, linestyle=linestyle, linewidth=2, label=name)
ax.set(title='Test D: clear contamination baseline', xlabel='x', ylabel='y')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()
print('Clear-contamination baseline (Huber vs IRLS; one-step vs IRLS):')
print(baseline_scores.to_string())

# Decrease outlier distance while keeping ratio and layout fixed.
distances = np.linspace(14, 0.5, 12)
records = []
for distance in distances:
    for seed in range(25):
        x_train, y_train, x_test, y_test, _ = make_data(ratio=baseline_ratio, distance=distance, layout='one_sided', seed=seed)
        for name, fit in methods.items():
            mse, parameter_error = metrics(fit(x_train, y_train), x_test, y_test)
            records.append((distance, name, mse, parameter_error))
d = pd.DataFrame(records, columns=['distance', 'method', 'test_mse', 'parameter_error']).groupby(['distance', 'method']).mean().reset_index()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
styles = {'Huber': ('tab:orange', '--'), 'Iterative reweighting': ('tab:green', '-.'), 'One-step reweighting': ('tab:red', ':')}
for name, fit in robust_methods.items():
    group = d[d.method == name].sort_values('distance')
    color, linestyle = styles[name]
    axes[0].plot(group.distance, group.test_mse, marker='o', color=color, linestyle=linestyle, linewidth=2.5, label=name)
    axes[1].plot(group.distance, group.parameter_error, marker='o', color=color, linestyle=linestyle, linewidth=2.5, label=name)
axes[0].set(xlabel='Outlier distance', ylabel='Test MSE', title='Test D')
axes[1].set(xlabel='Outlier distance', ylabel='Parameter error', title='Test D')
axes[0].legend()
axes[1].legend()
plt.tight_layout()
plt.show()
print('Test D distance-scan means over 25 random seeds:')
# Keep the OLS values in the numerical comparison, separate from the plot.
d.pivot(index='distance', columns='method', values=['test_mse', 'parameter_error'])


In [ ]:
# Test E: compare a single-line assumption with the two-line model
x1, y1, x2, y2, beta1, beta2 = make_two_lines(seed=8)
x = np.concatenate([x1, x2])
y = np.concatenate([y1, y2])
grid = np.linspace(-3.2, 3.2, 200)
single_fits = {name: fit(x, y) for name, fit in methods.items()}
two_line_fits = (ols_fit(x1, y1), ols_fit(x2, y2))

# Figure 1: show the true lines, all single-line fits, and the two separate fitted lines.
fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(x1, y1, s=18, alpha=0.45, color='tab:blue', label='Structure 1 data')
ax.scatter(x2, y2, s=18, alpha=0.45, color='tab:orange', label='Structure 2 data')
ax.plot(grid, predict(beta1, grid), color='tab:blue', linewidth=2, label='True line 1: y=x+2')
ax.plot(grid, predict(beta2, grid), color='tab:orange', linewidth=2, label='True line 2: y=-2x+8')
styles = {'OLS': ('black', '-'), 'Huber': ('tab:red', '--'), 'Iterative reweighting': ('tab:green', '-.'), 'One-step reweighting': ('tab:purple', ':')}
for name, beta in single_fits.items():
    color, linestyle = styles[name]
    ax.plot(grid, predict(beta, grid), color=color, linestyle=linestyle, linewidth=2.5, label=name)
ax.plot(grid, predict(two_line_fits[0], grid), color='navy', linestyle=':', linewidth=3, label='Two-line fit 1')
ax.plot(grid, predict(two_line_fits[1], grid), color='darkorange', linestyle=':', linewidth=3, label='Two-line fit 2')
ax.set(title='Test E: single-line versus two-line fitting', xlabel='x', ylabel='y')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

# Use clean held-out observations from both true lines for a fair five-method comparison.
test_rng = np.random.default_rng(108)
x1_test = test_rng.uniform(-3, 3, 1000)
x2_test = test_rng.uniform(-3, 3, 1000)
y1_test = predict(beta1, x1_test) + test_rng.normal(0, 0.35, 1000)
y2_test = predict(beta2, x2_test) + test_rng.normal(0, 0.35, 1000)
def group_scores(fits):
    mse = np.mean([np.mean((predict(fits[0], x1_test) - y1_test) ** 2), np.mean((predict(fits[1], x2_test) - y2_test) ** 2)])
    parameter_error = np.mean([np.linalg.norm(fits[0] - beta1), np.linalg.norm(fits[1] - beta2)])
    return mse, parameter_error
error_rows = []
for name, beta in single_fits.items():
    error_rows.append((name, *group_scores((beta, beta))))
error_rows.append(('Two-line fitting', *group_scores(two_line_fits)))
e = pd.DataFrame(error_rows, columns=['method', 'test_mse', 'parameter_error']).set_index('method')

# Figure 2: compare the two errors; the two-line values aggregate its two separate fits.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
e['test_mse'].plot.bar(ax=axes[0], color='tab:blue', rot=20)
e['parameter_error'].plot.bar(ax=axes[1], color='tab:orange', rot=20)
axes[0].set(title='Test E: clean test MSE', ylabel='MSE')
axes[1].set(title='Test E: mean parameter error', ylabel='Euclidean error')
plt.tight_layout()
plt.show()
print('Test E uses fixed training seed 8 and test seed 108:')
e
